In [100]:
import os
import json
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import requests


In [101]:
requests.get("http://localhost:11434").content
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")




# to use openai api key from .env file, uncomment the following lines and make sure you have a .env file with OPENAI_API_KEY set
# load_dotenv(override=True)
# api_key = os.getenv("OPENAI_API_KEY")

# if not api_key:
#     print("Error: OPENAI_API_KEY environment variable is not set.")
# elif not api_key.startswith("sk-"):
#     print("Error: OPENAI_API_KEY does not appear to be a valid key.")
# elif api_key.strip() != api_key:
#     print("Error: OPENAI_API_KEY contains leading or trailing whitespace.") 
# else: 
#     print("OPENAI_API_KEY is set correctly.")
             

In [102]:
system_message = """
You are an helpful assistant for an Airline called FlightAI
Give short continuous answers not moer than 1 line
Always be accurate, if you dont know the answer, say so"""

In [104]:
def chat(message, history):
    history = [{"role": h["role"], "content" : h["content"]} for h in history]
    messages = [{"role":"system", "content":system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(
        model = "qwen3:14b",
        messages= messages
    )
    return response.choices[0].message.content

gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.


In [105]:
ticket_prices = {"london": "$499", "paris": "$899", "berlin": "$299", "tokyo": "$899"}


def get_ticket_prices(destination_city):
    city = (destination_city or "").strip().lower()
    price = ticket_prices.get(city, "Unknown city")
    return f"The price of the ticket to {destination_city} is {price}"


In [106]:
print(get_ticket_prices("tokyo"))

The price of the ticket to tokyo is $899


In [107]:
price_function = {
    "name": "get_ticket_prices",
    "description": "Get the price of a return ticket to a destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


In [108]:
tools =[{"type": "function", "function" : price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_prices',
   'description': 'Get the price of a return ticket to a destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [110]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model="qwen3:14b", messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = ollama.chat.completions.create(model="qwen3:14b", messages=messages)
    
    return response.choices[0].message.content

In [111]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_prices":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get("destination_city")
        price_details = get_ticket_prices(city)
        return {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return None


In [ ]:
gr.ChatInterface(chat).launch()


#llama 3.2 calling every response with tool_calls, so input destination city is not provided and hence fails
#gemma3 does not support tools
#gpt-oss also failing with complex questions
#Qwen3:14b works well with tool calling


* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.
